# File Manager Notebook

| Cell | Operation |
|------|-----------|
| **1** | Install & Imports |
| **2** | All helper functions (run once) |
| **3** | ▶ Extract / Unzip any archive |
| **4** | ▶ Move files or folders |
| **5** | ▶ Copy files or folders |
| **6** | ▶ Rename a file or folder |
| **7** | ▶ Delete files or folders |
| **A** | Appendix — Browse local directory & disk usage |

## 1 — Install & Imports

In [1]:
%pip install lz4 py7zr zstandard tqdm --quiet

# System dependency for split/multi-disk zip files (run once):
import subprocess, shutil
if not shutil.which("7z"):
    subprocess.run(["sudo", "apt-get", "install", "-y", "p7zip-full"], check=True)
    print("✓ p7zip-full installed")
else:
    print("✓ 7z CLI already available")

Note: you may need to restart the kernel to use updated packages.
✓ 7z CLI already available


In [2]:
import os
import shutil
import subprocess
import tarfile
import zipfile
import gzip
import bz2
import lzma
import time
import threading
from pathlib import Path
from datetime import datetime

from tqdm import tqdm

print("✓ Core imports ready")

# ── Optional archive backends ──────────────────────────────────────────────
_lz4_ok = _zstd_ok = _7z_ok = False
try:
    import lz4.frame as lz4frame
    _lz4_ok = True;  print("✓ lz4  (handles .lz4, .tar.lz4)")
except ImportError:
    print("✗ lz4  — run: pip install lz4")

try:
    import zstandard as zstd
    _zstd_ok = True; print("✓ zstandard  (handles .zst, .tar.zst)")
except ImportError:
    print("✗ zstandard — run: pip install zstandard")

try:
    import py7zr
    _7z_ok = True;   print("✓ py7zr  (handles .7z)")
except ImportError:
    print("✗ py7zr — run: pip install py7zr")

# Check for system 7z CLI (handles split/multi-disk .zip natively)
_7z_cli_ok = shutil.which("7z") is not None
if _7z_cli_ok:
    print("✓ 7z CLI (handles split/multi-disk .zip, .7z)")
else:
    print("✗ 7z CLI not found — run: sudo apt-get install p7zip-full")

✓ Core imports ready
✓ lz4  (handles .lz4, .tar.lz4)
✓ zstandard  (handles .zst, .tar.zst)
✓ py7zr  (handles .7z)
✓ 7z CLI (handles split/multi-disk .zip, .7z)


## 2 — Helper Functions

Run this cell once. All operation cells below depend on it.

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _human_size(n: int) -> str:
    for u in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


def _box(lines: list[str], width: int = 70) -> str:
    sep = "─" * (width - 2)
    def row(s): return f"║  {s:<{width-4}}║"
    out = [f"╔{sep}╗"]
    for l in lines:
        out.append(f"╠{sep}╣" if l == "---" else row(l))
    out.append(f"╚{sep}╝")
    return "\n".join(out)


def _confirm(prompt: str) -> bool:
    """Ask the user to type 'yes' to confirm a destructive operation."""
    ans = input(f"  {prompt} [yes / no]: ").strip().lower()
    return ans == "yes"


def _archive_type(path: Path) -> str:
    """
    Detect archive type from the file name.
    Returns one of: tar.lz4 | tar.zst | tar.gz | tar.bz2 | tar.xz | tar
                    zip | gz | bz2 | xz | lz4 | zst | 7z | rar | unknown
    """
    name = path.name.lower()
    for ext in (".tar.lz4", ".tar.zst", ".tar.gz", ".tar.bz2",
                ".tar.xz", ".tar.lzma", ".tgz", ".tbz2",
                ".zip", ".7z", ".rar",
                ".gz", ".bz2", ".xz", ".lzma", ".lz4", ".zst", ".tar"):
        if name.endswith(ext):
            return ext.lstrip(".")
    return "unknown"


# ── Extract ────────────────────────────────────────────────────────────────

def _extract_tar_lz4(src: Path, dest: Path):
    """Stream-decompress LZ4 → pipe into tarfile → extract with progress."""
    if not _lz4_ok:
        raise RuntimeError("lz4 not installed. Run: pip install lz4")

    file_size = src.stat().st_size
    dest.mkdir(parents=True, exist_ok=True)

    print(f"Decompressing (LZ4) and extracting …")
    with (
        open(src, "rb") as raw,
        lz4frame.open(raw, mode="rb") as lz4_stream,
        tqdm(total=file_size, unit="B", unit_scale=True,
             unit_divisor=1024, desc="Reading") as bar,
    ):
        class _TrackedStream:
            """Wraps the lz4 stream to update the progress bar on reads."""
            def read(self, n=-1):
                chunk = lz4_stream.read(n)
                bar.update(raw.tell() - bar.n)
                return chunk
            def readinto(self, b):
                n = lz4_stream.readinto(b)
                bar.update(raw.tell() - bar.n)
                return n
            # tarfile needs these attributes
            readable  = lambda s: True
            writable  = lambda s: False
            seekable  = lambda s: False
            tell      = lambda s: lz4_stream.tell()

        with tarfile.open(fileobj=_TrackedStream(), mode="r|") as tf:
            tf.extractall(path=dest)


def _extract_tar_zst(src: Path, dest: Path):
    """Stream-decompress Zstandard → tarfile."""
    if not _zstd_ok:
        raise RuntimeError("zstandard not installed. Run: pip install zstandard")
    file_size = src.stat().st_size
    dest.mkdir(parents=True, exist_ok=True)
    dctx = zstd.ZstdDecompressor()
    with open(src, "rb") as fh:
        with tqdm(total=file_size, unit="B", unit_scale=True,
                  unit_divisor=1024, desc="Reading") as bar:
            stream = dctx.stream_reader(fh)
            with tarfile.open(fileobj=stream, mode="r|") as tf:
                for member in tf:
                    tf.extract(member, path=dest)
                    bar.update(fh.tell() - bar.n)


def _7z_archive_info(src: Path) -> dict:
    """Quick probe with '7z l' to get total size and file count."""
    info = {"files": 0, "uncompressed": 0, "volumes": 1}
    try:
        out = subprocess.run(
            ["7z", "l", str(src)], capture_output=True, text=True, timeout=30,
        )
        for line in out.stdout.splitlines():
            if "files" in line and line.strip()[0].isdigit():
                parts = line.split()
                if len(parts) >= 3:
                    try:
                        info["uncompressed"] = int(parts[0])
                    except ValueError:
                        pass
                    for p in parts:
                        if p.isdigit():
                            info["files"] = int(p)
            if "Volumes:" in line:
                try:
                    info["volumes"] = int(line.split(":")[-1].strip())
                except ValueError:
                    pass
    except Exception:
        pass
    return info


def _extract_zip_7z_cli(src: Path, dest: Path):
    """Fallback for split/multi-disk zip files using the system 7z CLI."""
    if not globals().get("_7z_cli_ok") and not shutil.which("7z"):
        raise RuntimeError(
            "7z CLI not found. Install with: sudo apt-get install p7zip-full"
        )
    dest.mkdir(parents=True, exist_ok=True)

    info = _7z_archive_info(src)
    vol_str = f" ({info['volumes']} volumes)" if info["volumes"] > 1 else ""
    total_files = info["files"]
    total_size  = info["uncompressed"]
    if total_size:
        print(f"  ↳ Multi-disk zip{vol_str} — {_human_size(total_size)} "
              f"uncompressed, {total_files:,} files")
    else:
        print(f"  ↳ Multi-disk zip detected{vol_str}")
    print(f"  ↳ Using 7z CLI …")

    def _fmt_t(sec):
        if sec < 60:
            return f"{sec:.0f}s"
        m, s = divmod(int(sec), 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"

    cmd = ["7z", "x", str(src), f"-o{dest}", "-y", "-bsp1", "-bb0"]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    buf = b""
    files_seen = 0
    last_pct_num = 0
    t0 = time.time()

    while True:
        chunk = proc.stdout.read(1)
        if not chunk:
            break
        if chunk in (b"\r", b"\n"):
            line = buf.decode("utf-8", errors="replace").strip()
            buf = b""
            if not line:
                continue
            if line and line[0].isdigit() and "%" in line[:5]:
                try:
                    pct_num = int(line.split("%")[0].strip())
                except ValueError:
                    continue
                if pct_num != last_pct_num and pct_num > 0:
                    last_pct_num = pct_num
                    elapsed = time.time() - t0
                    eta = (elapsed / pct_num) * (100 - pct_num)
                    bar_len = 30
                    filled  = int(bar_len * pct_num / 100)
                    bar_str = "█" * filled + "░" * (bar_len - filled)
                    status  = (f"\r  7z: {pct_num:>3}% |{bar_str}| "
                               f"elapsed {_fmt_t(elapsed)}  "
                               f"ETA {_fmt_t(eta)}  "
                               f"({files_seen:,} files)")
                    print(status, end="", flush=True)
            elif line.startswith("- "):
                files_seen += 1
            elif ("Everything is Ok" in line or "Extracting archive" in line
                  or "Scanning" in line):
                print(f"\n  {line}")
        else:
            buf += chunk
    proc.wait()
    elapsed = round(time.time() - t0, 1)
    if last_pct_num:
        print()
    print(f"  7z done — {files_seen:,} files in {_fmt_t(elapsed)}")
    if proc.returncode != 0:
        raise RuntimeError(f"7z exited with code {proc.returncode} for {src.name}")


def _extract_zip(src: Path, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(src) as zf:
            members = zf.infolist()
            total   = sum(m.file_size for m in members)
            with tqdm(total=total, unit="B", unit_scale=True,
                      unit_divisor=1024, desc="Extracting") as bar:
                for member in members:
                    zf.extract(member, path=dest)
                    bar.update(member.file_size)
    except zipfile.BadZipFile as e:
        if "span multiple disks" in str(e):
            _extract_zip_7z_cli(src, dest)
        else:
            raise


def _extract_7z(src: Path, dest: Path):
    if not _7z_ok:
        raise RuntimeError("py7zr not installed. Run: pip install py7zr")
    dest.mkdir(parents=True, exist_ok=True)
    with py7zr.SevenZipFile(src, mode="r") as archive:
        file_list = archive.getnames()
        print(f"  Extracting {len(file_list)} items …")
        archive.extractall(path=dest)


def _extract_single(src: Path, dest_file: Path, open_fn):
    """Decompress a single-file archive (.gz, .bz2, .xz, .lz4, .zst)."""
    file_size = src.stat().st_size
    dest_file.parent.mkdir(parents=True, exist_ok=True)
    with open_fn(src, "rb") as f_in, open(dest_file, "wb") as f_out:
        with tqdm(total=file_size, unit="B", unit_scale=True,
                  unit_divisor=1024, desc="Decompressing") as bar:
            while chunk := f_in.read(1 << 20):  # 1 MB chunks
                f_out.write(chunk)
                bar.update(len(chunk))


def extract(
    source_path: str | Path,
    extract_to:  str | Path,
    overwrite:   bool = False,
) -> Path:
    """
    Extract / decompress any archive into *extract_to*.

    Supported formats
    -----------------
    .tar.lz4  .tar.zst  .tar.gz  .tar.bz2  .tar.xz  .tar
    .zip  .7z  .gz  .bz2  .xz  .lz4  .zst

    Parameters
    ----------
    source_path : Path to the archive file.
    extract_to  : Directory to extract into.
    overwrite   : If True, remove existing destination before extracting.
    """
    src  = Path(source_path)
    dest = Path(extract_to)

    if not src.exists():
        raise FileNotFoundError(f"Archive not found: {src}")

    atype = _archive_type(src)
    if atype == "unknown":
        raise ValueError(f"Unsupported archive type: {src.name}")

    if dest.exists() and not overwrite:
        print(_box([
            "EXTRACT SKIPPED", "---",
            f"Archive  : {src.name}",
            f"Dest     : {dest}",
            "---",
            "⏭  Destination already exists — set OVERWRITE = True to re-extract",
        ]))
        return dest

    t0 = time.time()
    print(f"Archive  : {src.name}  ({_human_size(src.stat().st_size)})")
    print(f"Format   : {atype}")
    print(f"Dest     : {dest}\n")

    if atype == "tar.lz4":
        _extract_tar_lz4(src, dest)
    elif atype == "tar.zst":
        _extract_tar_zst(src, dest)
    elif atype in ("tar.gz", "tgz", "tar.bz2", "tbz2", "tar.xz",
                   "tar.lzma", "tar"):
        dest.mkdir(parents=True, exist_ok=True)
        with tarfile.open(src) as tf:
            members = tf.getmembers()
            with tqdm(total=len(members), unit="file", desc="Extracting") as bar:
                for m in members:
                    tf.extract(m, path=dest)
                    bar.update(1)
    elif atype == "zip":
        _extract_zip(src, dest)
    elif atype == "7z":
        _extract_7z(src, dest)
    elif atype == "gz":
        _extract_single(src, dest / src.stem, gzip.open)
    elif atype == "bz2":
        _extract_single(src, dest / src.stem, bz2.open)
    elif atype in ("xz", "lzma"):
        _extract_single(src, dest / src.stem, lzma.open)
    elif atype == "lz4":
        if not _lz4_ok:
            raise RuntimeError("pip install lz4")
        _extract_single(src, dest / src.stem, lz4frame.open)
    elif atype == "zst":
        if not _zstd_ok:
            raise RuntimeError("pip install zstandard")
        dctx = zstd.ZstdDecompressor()
        _extract_single(src, dest / src.stem,
                        lambda p, m: dctx.stream_reader(open(p, "rb")))

    elapsed  = time.time() - t0
    out_size = sum(f.stat().st_size for f in dest.rglob("*") if f.is_file())
    n_files  = sum(1 for f in dest.rglob("*") if f.is_file())
    n_dirs   = sum(1 for f in dest.rglob("*") if f.is_dir())

    print(_box([
        "EXTRACTION COMPLETE", "---",
        f"Archive      : {src.name}",
        f"Format       : {atype}",
        f"Extracted to : {dest}",
        "---",
        f"Files        : {n_files}",
        f"Sub-folders  : {n_dirs}",
        f"Total size   : {_human_size(out_size)}",
        f"Time taken   : {elapsed:.1f}s",
    ]))
    return dest


# ── Move ───────────────────────────────────────────────────────────────────

def move(
    source_path: str | Path,
    dest_path:   str | Path,
    overwrite:   bool = False,
) -> Path:
    """
    Move a file or folder to *dest_path*.

    If *dest_path* is an existing directory the item is moved inside it.
    Set overwrite=True to replace an existing destination file.
    """
    src  = Path(source_path)
    dest = Path(dest_path)
    if not src.exists():
        raise FileNotFoundError(f"Not found: {src}")

    final = dest / src.name if dest.is_dir() else dest
    if final.exists() and not overwrite:
        raise FileExistsError(
            f"Destination exists: {final}\nSet overwrite=True to replace it.")
    if final.exists():
        shutil.rmtree(final) if final.is_dir() else final.unlink()

    final.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(final))

    kind = "folder" if final.is_dir() else "file"
    print(_box([
        "MOVE COMPLETE", "---",
        f"Type   : {kind}",
        f"From   : {src}",
        f"To     : {final}",
    ]))
    return final


# ── Copy ───────────────────────────────────────────────────────────────────

def copy(
    source_path: str | Path,
    dest_path:   str | Path,
    overwrite:   bool = False,
) -> Path:
    """
    Copy a file or entire folder tree to *dest_path*.
    If *dest_path* is an existing directory the item is copied inside it.
    """
    src  = Path(source_path)
    dest = Path(dest_path)
    if not src.exists():
        raise FileNotFoundError(f"Not found: {src}")

    final = dest / src.name if dest.is_dir() else dest
    if final.exists() and not overwrite:
        raise FileExistsError(
            f"Destination exists: {final}\nSet overwrite=True to replace it.")
    if final.exists():
        shutil.rmtree(final) if final.is_dir() else final.unlink()

    t0 = time.time()
    if src.is_dir():
        all_files = [f for f in src.rglob("*") if f.is_file()]
        total     = sum(f.stat().st_size for f in all_files)
        with tqdm(total=total, unit="B", unit_scale=True,
                  unit_divisor=1024, desc="Copying") as bar:
            def _copy_fn(s, d):
                r = shutil.copy2(s, d)
                bar.update(Path(s).stat().st_size)
                return r
            shutil.copytree(src, final, copy_function=_copy_fn)
        kind = "folder"
        size = _human_size(sum(f.stat().st_size for f in final.rglob("*") if f.is_file()))
    else:
        final.parent.mkdir(parents=True, exist_ok=True)
        size = _human_size(src.stat().st_size)
        with tqdm(total=src.stat().st_size, unit="B", unit_scale=True,
                  unit_divisor=1024, desc="Copying") as bar:
            with open(src, "rb") as f_in, open(final, "wb") as f_out:
                while chunk := f_in.read(1 << 20):
                    f_out.write(chunk)
                    bar.update(len(chunk))
        kind = "file"

    elapsed = time.time() - t0
    print(_box([
        "COPY COMPLETE", "---",
        f"Type   : {kind}",
        f"From   : {src}",
        f"To     : {final}",
        f"Size   : {size}",
        f"Time   : {elapsed:.1f}s",
    ]))
    return final


# ── Rename ─────────────────────────────────────────────────────────────────

def rename(
    source_path: str | Path,
    new_name:    str,
) -> Path:
    """
    Rename a file or folder in-place (same parent directory).

    Parameters
    ----------
    source_path : Current path of the file or folder.
    new_name    : New name only (not a full path), e.g. "dataset_v2".
    """
    src   = Path(source_path)
    final = src.parent / new_name
    if not src.exists():
        raise FileNotFoundError(f"Not found: {src}")
    if final.exists():
        raise FileExistsError(f"Already exists: {final}")
    src.rename(final)
    print(_box([
        "RENAME COMPLETE", "---",
        f"From : {src.name}",
        f"To   : {final.name}",
        f"Dir  : {src.parent}",
    ]))
    return final


# ── Delete ─────────────────────────────────────────────────────────────────

def delete(
    target_path: str | Path,
    confirm:     bool = True,
) -> bool:
    """
    Delete a file or folder permanently.

    Parameters
    ----------
    target_path : Path to delete.
    confirm     : If True (default), asks for interactive confirmation.
                  Set to False only when you are certain — no undo!
    """
    target = Path(target_path)
    if not target.exists():
        print(f"  ⚠  Not found (nothing deleted): {target}")
        return False

    kind = "folder" if target.is_dir() else "file"
    if target.is_dir():
        n     = sum(1 for _ in target.rglob("*") if _.is_file())
        size  = _human_size(sum(f.stat().st_size for f in target.rglob("*") if f.is_file()))
        label = f"{n} files — {size}"
    else:
        label = _human_size(target.stat().st_size)

    print(f"  ⚠  About to permanently delete {kind}: {target}")
    print(f"     Contains: {label}")

    if confirm and not _confirm("Type 'yes' to confirm deletion"):
        print("  ✗  Cancelled.")
        return False

    shutil.rmtree(target) if target.is_dir() else target.unlink()
    print(_box([
        "DELETE COMPLETE", "---",
        f"Deleted ({kind}) : {target}",
        f"Content          : {label}",
    ]))
    return True


# ── Browse / disk usage ────────────────────────────────────────────────────

def browse(
    path:      str | Path = ".",
    max_depth: int = 2,
    show_hidden: bool = False,
) -> None:
    """
    Print a tree view of *path* up to *max_depth* levels deep.
    Shows file sizes and last-modified timestamps.
    """
    root = Path(path)
    if not root.exists():
        print(f"Path not found: {root}")
        return

    def _tree(p: Path, prefix: str, depth: int):
        if depth > max_depth:
            return
        items = sorted(p.iterdir(), key=lambda x: (x.is_file(), x.name))
        if not show_hidden:
            items = [i for i in items if not i.name.startswith(".")]
        for idx, item in enumerate(items):
            is_last  = idx == len(items) - 1
            branch   = "└── " if is_last else "├── "
            ext_pre  = "    " if is_last else "│   "
            if item.is_dir():
                n_ch = sum(1 for _ in item.iterdir())
                print(f"{prefix}{branch}📁 {item.name}/  [{n_ch} items]")
                _tree(item, prefix + ext_pre, depth + 1)
            else:
                mtime = datetime.fromtimestamp(item.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
                size  = _human_size(item.stat().st_size)
                print(f"{prefix}{branch}📄 {item.name}  {size:>10}  {mtime}")

    total_files = sum(1 for f in root.rglob("*") if f.is_file())
    total_size  = sum(f.stat().st_size for f in root.rglob("*") if f.is_file())
    print(f"📂 {root}")
    print(f"   {total_files} files — {_human_size(total_size)} total\n")
    _tree(root, "", 1)


def disk_usage(path: str | Path = ".") -> None:
    """Print disk usage for every immediate child of *path*, sorted by size."""
    root = Path(path)
    if not root.exists():
        print(f"Path not found: {root}")
        return

    rows = []
    for item in sorted(root.iterdir()):
        if item.is_file():
            rows.append((item.stat().st_size, "file",   item.name))
        elif item.is_dir():
            sz = sum(f.stat().st_size for f in item.rglob("*") if f.is_file())
            rows.append((sz, "folder", item.name))

    rows.sort(reverse=True)
    total = sum(r[0] for r in rows)
    print(f"\n{'Name':<45} {'Type':<8} {'Size':>12}  {'%':>6}")
    print("─" * 78)
    for sz, kind, name in rows:
        pct = sz / total * 100 if total else 0
        icon = "📁" if kind == "folder" else "📄"
        print(f"{icon} {name:<43} {kind:<8} {_human_size(sz):>12}  {pct:5.1f}%")
    print("─" * 78)
    print(f"{'Total':<45} {'':8} {_human_size(total):>12}")


print("✓ All helpers loaded — ready to use")

✓ All helpers loaded — ready to use


## 3 — Extract / Unzip

Handles: `.tar.lz4` `.tar.zst` `.tar.gz` `.tar.bz2` `.tar.xz` `.tar` `.zip` `.7z` `.gz` `.bz2` `.xz` `.lz4` `.zst`

In [ ]:
# # ── INPUT ──────────────────────────────────────────────────────────────────
# SOURCE_ARCHIVE = "/home/taiaburrahman/dataset_manager_pro/data/wasabi/gen_ai_detector_dataset_scaled_224/human.tar.lz4"
# EXTRACT_TO     = "/home/taiaburrahman/dataset_manager_pro/data/gen_ai_detector_dataset_scaled_224/human/"
# OVERWRITE      = False   # True = remove existing dest and re-extract
# # ───────────────────────────────────────────────────────────────────────────

# extract(SOURCE_ARCHIVE, EXTRACT_TO, overwrite=OVERWRITE)

## 3b — Batch Extract (Whole Folder)

Recursively finds **every archive** inside `SOURCE_FOLDER` (and all sub-folders) and extracts each one.

| Option | Meaning |
|---|---|
| `SOURCE_FOLDER` | Root folder to search for archives |
| `EXTRACT_TO` | Where to extract each archive. `None` = extract next to the archive file |
| `SAME_STRUCTURE` | `True` = mirror the source folder tree inside `EXTRACT_TO`; `False` = all go flat into `EXTRACT_TO` |
| `OVERWRITE` | Re-extract even if the destination already exists |

In [7]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_FOLDER  = "/home/taiaburrahman/dataset_manager_pro/data/wasabi/GenAI_Image_Database/Generated_Data_2025"
EXTRACT_TO     = "/home/taiaburrahman/dataset_manager_pro/data/processed/gen_ai_detector_dataset/GenAI_Image_Database/Generated_Data_2025"
SAME_STRUCTURE = True        # True  → mirror folder tree inside EXTRACT_TO
OVERWRITE      = True        # True = re-extract even if dest already exists
SAVE_REPORT    = True        # save JSON + TXT report next to SOURCE_FOLDER
# ───────────────────────────────────────────────────────────────────────────

import json as _json
from datetime import datetime

ARCHIVE_EXTS = {".lz4", ".zst", ".gz", ".bz2", ".xz", ".zip", ".7z", ".tar","tar.gz"}

def _is_archive(p: Path) -> bool:
    name = p.name.lower()
    for ext in ARCHIVE_EXTS:
        if name.endswith(ext):
            return True
    return False

def _fmt_time(sec):
    if sec < 60:
        return f"{sec:.0f}s"
    m, s = divmod(int(sec), 60)
    if m < 60:
        return f"{m}m{s:02d}s"
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}m"

root = Path(SOURCE_FOLDER)
archives = sorted([f for f in root.rglob("*") if f.is_file() and _is_archive(f)])
archive_sizes = [a.stat().st_size for a in archives]
total_size = sum(archive_sizes)

print(f"Source folder : {root}")
print(f"Extract to    : {EXTRACT_TO or '(beside each archive)'}")
print(f"Archives found: {len(archives)}  ({_human_size(total_size)} total)\n")
print(f"{'#':>3}  {'Archive':<50}  {'Size':>10}")
print(f"{'─'*70}")
for i, a in enumerate(archives, 1):
    print(f"{i:>3}  {str(a.relative_to(root)):<50}  {_human_size(a.stat().st_size):>10}")
print(f"{'─'*70}")

print()

# ── Process each archive ───────────────────────────────────────────────────
results_log = []
t_start_all = time.time()
bytes_done = 0

for idx, archive in enumerate(archives, 1):
    arc_size = archive_sizes[idx - 1]

    # Size-weighted progress display
    pct = (bytes_done / total_size * 100) if total_size else 0
    elapsed_all = time.time() - t_start_all
    if bytes_done > 0 and total_size > 0:
        eta = elapsed_all * (total_size - bytes_done) / bytes_done
    else:
        eta = 0
    bar_len = 30
    filled  = int(bar_len * pct / 100)
    bar_str = "█" * filled + "░" * (bar_len - filled)
    print(f"\n{'─'*80}")
    print(f"  Progress: {pct:5.1f}% |{bar_str}| "
          f"{idx}/{len(archives)} files  "
          f"({_human_size(bytes_done)}/{_human_size(total_size)})  "
          f"elapsed {_fmt_time(elapsed_all)}  ETA {_fmt_time(eta)}")
    print(f"{'─'*80}")

    if EXTRACT_TO is None:
        out_dir = archive.parent
    elif SAME_STRUCTURE:
        rel     = archive.parent.relative_to(root)
        out_dir = Path(EXTRACT_TO) / rel
    else:
        out_dir = Path(EXTRACT_TO)

    entry = {
        "index":         idx,
        "archive":       str(archive.relative_to(root)),
        "archive_name":  archive.name,
        "archive_size":  arc_size,
        "extract_to":    str(out_dir),
        "status":        "",
        "elapsed_sec":   0,
        "files_extracted": 0,
        "extracted_size":  0,
        "error":         "",
    }

    t0 = time.time()
    try:
        result = extract(archive, out_dir, overwrite=OVERWRITE)
        entry["elapsed_sec"] = round(time.time() - t0, 1)
        if result is None:
            entry["status"] = "skipped"
        else:
            entry["status"] = "done"
            if result.is_dir():
                entry["files_extracted"] = sum(1 for f in result.rglob("*") if f.is_file())
                entry["extracted_size"]  = sum(f.stat().st_size for f in result.rglob("*") if f.is_file())
    except Exception as e:
        entry["elapsed_sec"] = round(time.time() - t0, 1)
        entry["status"] = "failed"
        entry["error"]  = str(e)
        print(f"\n  ✗ {archive.name}: {e}")

    bytes_done += arc_size

    sym = {"done": "✓", "skipped": "⏭", "failed": "✗"}.get(entry["status"], "?")
    print(f"  {sym} [{idx}/{len(archives)}] {archive.name}  "
          f"{_human_size(entry['archive_size'])} → "
          f"{entry['files_extracted']:,} files  "
          f"({_fmt_time(entry['elapsed_sec'])})  {entry['status']}")

    results_log.append(entry)

total_elapsed = round(time.time() - t_start_all, 1)

# Final 100% progress line
print(f"\n{'─'*80}")
print(f"  Progress: 100.0% |{'█'*30}| "
      f"{len(archives)}/{len(archives)} files  "
      f"({_human_size(total_size)}/{_human_size(total_size)})  "
      f"elapsed {_fmt_time(total_elapsed)}  DONE")
print(f"{'─'*80}")

# ── Count stats ────────────────────────────────────────────────────────────
n_done    = sum(1 for r in results_log if r["status"] == "done")
n_skipped = sum(1 for r in results_log if r["status"] == "skipped")
n_failed  = sum(1 for r in results_log if r["status"] == "failed")

# ── Display final results table ────────────────────────────────────────────
_sym = {"done": "✓", "skipped": "⏭", "failed": "✗"}

print(f"\n{'─'*100}")
print(f" {'#':>3}  {'St':^4}  {'Archive':<42}  {'Arch Size':>10}  {'Extracted':>10}  {'Files':>8}  {'Time':>8}")
print(f"{'─'*100}")
for r in results_log:
    sym = _sym.get(r["status"], "?")
    sz  = _human_size(r["archive_size"])
    exsz = _human_size(r.get("extracted_size", 0)) if r.get("extracted_size") else "—"
    tm  = _fmt_time(r["elapsed_sec"]) if r["elapsed_sec"] else "—"
    fc  = f"{r.get('files_extracted', 0):,}" if r.get("files_extracted") else "—"
    print(f" {r['index']:>3}  {sym:^4}  {r['archive']:<42}  {sz:>10}  {exsz:>10}  {fc:>8}  {tm:>8}")
    if r["error"]:
        print(f"      └─ {r['error'][:85]}")
print(f"{'─'*100}")

# ── Summary box ────────────────────────────────────────────────────────────
W = 68
sep = "─" * (W - 2)
elapsed_str = f"{int(total_elapsed//60)}m {int(total_elapsed%60)}s" if total_elapsed >= 60 else f"{total_elapsed}s"

print(f"\n╔{sep}╗")
print(f"║  {'BATCH EXTRACT COMPLETE':<{W-4}}║")
print(f"╠{sep}╣")
print(f"║  Source folder  : {str(root)[:W-20]:<{W-20}}║")
rt = str(EXTRACT_TO)[:W-20] if EXTRACT_TO else "(beside archive)"
print(f"║  Extract to     : {rt:<{W-20}}║")
print(f"║  Total archives : {len(archives):<{W-20}}║")
print(f"║  ✓ Done         : {n_done:<{W-20}}║")
print(f"║  ⏭ Skipped      : {n_skipped:<{W-20}}║")
print(f"║  ✗ Failed       : {n_failed:<{W-20}}║")
print(f"║  Total time     : {elapsed_str:<{W-20}}║")
print(f"╚{sep}╝")

# ── Save report ────────────────────────────────────────────────────────────
if SAVE_REPORT:
    report_dir = Path(EXTRACT_TO) if EXTRACT_TO else root
    report_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    report = {
        "timestamp":     datetime.now().isoformat(),
        "source_folder": str(root),
        "extract_to":    str(EXTRACT_TO),
        "overwrite":     OVERWRITE,
        "same_structure": SAME_STRUCTURE,
        "total_archives": len(archives),
        "done":          n_done,
        "skipped":       n_skipped,
        "failed":        n_failed,
        "total_elapsed_sec": total_elapsed,
        "files": results_log,
    }

    # JSON report
    json_path = report_dir / f"batch_extract_report_{ts}.json"
    with open(json_path, "w") as f:
        _json.dump(report, f, indent=2, default=str)

    # TXT summary
    txt_path = report_dir / f"batch_extract_report_{ts}.txt"
    with open(txt_path, "w") as f:
        f.write(f"BATCH EXTRACT REPORT — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"{'='*70}\n\n")
        f.write(f"Source folder : {root}\n")
        f.write(f"Extract to    : {EXTRACT_TO}\n")
        f.write(f"Overwrite     : {OVERWRITE}\n")
        f.write(f"Total archives: {len(archives)}\n")
        f.write(f"Done: {n_done}  |  Skipped: {n_skipped}  |  Failed: {n_failed}\n")
        f.write(f"Total time    : {elapsed_str}\n\n")

        f.write(f"{'#':>3}  {'Status':<8}  {'Archive':<45}  {'Size':>10}  {'Time':>8}  {'Files':>7}\n")
        f.write(f"{'-'*90}\n")
        for r in results_log:
            sym = _sym.get(r["status"], "?")
            sz  = _human_size(r["archive_size"])
            tm  = f"{r['elapsed_sec']}s" if r["elapsed_sec"] else "—"
            fc  = str(r.get("files_extracted", "—"))
            f.write(f"{r['index']:>3}  {sym:<8}  {r['archive']:<45}  {sz:>10}  {tm:>8}  {fc:>7}\n")
            if r["error"]:
                f.write(f"     └─ Error: {r['error']}\n")

        if n_failed:
            f.write(f"\n{'='*70}\nFAILED ARCHIVES:\n")
            for r in results_log:
                if r["status"] == "failed":
                    f.write(f"  {r['archive']}: {r['error']}\n")

    print(f"\n  📄 JSON report → {json_path}")
    print(f"  📄 TXT report  → {txt_path}")

Source folder : /home/taiaburrahman/dataset_manager_pro/data/wasabi/GenAI_Image_Database/Generated_Data_2025
Extract to    : /home/taiaburrahman/dataset_manager_pro/data/processed/gen_ai_detector_dataset/GenAI_Image_Database/Generated_Data_2025
Archives found: 14  (15.7 GB total)

  #  Archive                                                   Size
──────────────────────────────────────────────────────────────────────
  1  Gen_samples_DALLE_2_300325.tar.lz4                      4.5 GB
  2  Gen_samples_DALLE_3_300325.tar.lz4                      4.1 GB
  3  Gen_samples_GPT_IMAGE_1_070525.tar.lz4                469.4 MB
  4  Gen_samples_Gemini_100325.tar.lz4                     453.2 MB
  5  Gen_samples_Gemini_150325.tar.lz4                     329.2 MB
  6  Gen_samples_Gemini_190325.tar.gz                      356.9 MB
  7  Gen_samples_Gemini_270325.tar.lz4                     514.7 MB
  8  Gen_samples_Gemini_280325.tar.lz4                     657.6 MB
  9  Gen_samples_Grok2_200325.tar.g

Reading: 100%|██████████| 4.52G/4.52G [00:51<00:00, 94.7MB/s]


╔────────────────────────────────────────────────────────────────────╗
║  EXTRACTION COMPLETE                                               ║
╠────────────────────────────────────────────────────────────────────╣
║  Archive      : Gen_samples_DALLE_2_300325.tar.lz4                 ║
║  Format       : tar.lz4                                            ║
║  Extracted to : /home/taiaburrahman/dataset_manager_pro/data/processed/gen_ai_detector_dataset/GenAI_Image_Database/Generated_Data_2025║
╠────────────────────────────────────────────────────────────────────╣
║  Files        : 8645                                               ║
║  Sub-folders  : 5                                                  ║
║  Total size   : 6.1 GB                                             ║
║  Time taken   : 51.3s                                              ║
╚────────────────────────────────────────────────────────────────────╝
  ✓ [1/14] Gen_samples_DALLE_2_300325.tar.lz4  4.5 GB → 8,645 files  (52s)  done

Reading: 100%|██████████| 4.06G/4.06G [00:29<00:00, 148MB/s]


╔────────────────────────────────────────────────────────────────────╗
║  EXTRACTION COMPLETE                                               ║
╠────────────────────────────────────────────────────────────────────╣
║  Archive      : Gen_samples_DALLE_3_300325.tar.lz4                 ║
║  Format       : tar.lz4                                            ║
║  Extracted to : /home/taiaburrahman/dataset_manager_pro/data/processed/gen_ai_detector_dataset/GenAI_Image_Database/Generated_Data_2025║
╠────────────────────────────────────────────────────────────────────╣
║  Files        : 11204                                              ║
║  Sub-folders  : 6                                                  ║
║  Total size   : 10.1 GB                                            ║
║  Time taken   : 29.5s                                              ║
╚────────────────────────────────────────────────────────────────────╝
  ✓ [2/14] Gen_samples_DALLE_3_300325.tar.lz4  4.1 GB → 11,204 files  (30s)  don

...

╔────────────────────────────────────────────────────────────────────╗
║  EXTRACTION COMPLETE                                               ║
╠────────────────────────────────────────────────────────────────────╣
║  Archive      : Gen_samples_Grok3_220325.tar.gz                    ║
║  Format       : tar.gz                                             ║
║  Extracted to : /home/taiaburrahman/dataset_manager_pro/data/processed/gen_ai_detector_dataset/GenAI_Image_Database/Generated_Data_2025║
╠────────────────────────────────────────────────────────────────────╣
║  Files        : 66884                                              ║
║  Sub-folders  : 17                                                 ║
║  Total size   : 16.0 GB                                            ║
║  Time taken   : 6.1s                                               ║
╚────────────────────────────────────────────────────────────────────╝
  ✓ [12/14] Gen_samples_Grok3_220325.tar.gz  658.2 MB → 66,884 files  (10s)  don

Reading: 100%|██████████| 212M/212M [00:03<00:00, 71.9MB/s] 


╔────────────────────────────────────────────────────────────────────╗
║  EXTRACTION COMPLETE                                               ║
╠────────────────────────────────────────────────────────────────────╣
║  Archive      : selected_Real_Samples_phase_2.tar.lz4              ║
║  Format       : tar.lz4                                            ║
║  Extracted to : /home/taiaburrahman/dataset_manager_pro/data/processed/gen_ai_detector_dataset/GenAI_Image_Database/Generated_Data_2025║
╠────────────────────────────────────────────────────────────────────╣
║  Files        : 71030                                              ║
║  Sub-folders  : 18                                                 ║
║  Total size   : 16.2 GB                                            ║
║  Time taken   : 3.1s                                               ║
╚────────────────────────────────────────────────────────────────────╝
  ✓ [13/14] selected_Real_Samples_phase_2.tar.lz4  211.7 MB → 71,030 files  (7s)

Extracting: 100%|██████████| 4490/4490 [00:08<00:00, 501.78file/s]


╔────────────────────────────────────────────────────────────────────╗
║  EXTRACTION COMPLETE                                               ║
╠────────────────────────────────────────────────────────────────────╣
║  Archive      : selected_real_samples.tar.gz                       ║
║  Format       : tar.gz                                             ║
║  Extracted to : /home/taiaburrahman/dataset_manager_pro/data/processed/gen_ai_detector_dataset/GenAI_Image_Database/Generated_Data_2025║
╠────────────────────────────────────────────────────────────────────╣
║  Files        : 71030                                              ║
║  Sub-folders  : 18                                                 ║
║  Total size   : 16.2 GB                                            ║
║  Time taken   : 9.1s                                               ║
╚────────────────────────────────────────────────────────────────────╝
  ✓ [14/14] selected_real_samples.tar.gz  977.2 MB → 71,030 files  (13s)  done



## 4 — Move

Move a file **or** an entire folder. If `DEST_PATH` is an existing directory, the item is moved *inside* it.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/extracted/genAI_2/"
DEST_PATH   = "/home/taiaburrahman/dataset_manager_pro/data/ready/"
OVERWRITE   = False
# ───────────────────────────────────────────────────────────────────────────

move(SOURCE_PATH, DEST_PATH, overwrite=OVERWRITE)

## 5 — Copy

Copy a file **or** an entire folder tree with a byte-level progress bar.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/ready/genAI_2/"
DEST_PATH   = "/home/taiaburrahman/dataset_manager_pro/data/backup/"
OVERWRITE   = False
# ───────────────────────────────────────────────────────────────────────────

copy(SOURCE_PATH, DEST_PATH, overwrite=OVERWRITE)

## 6 — Rename

Rename a file or folder in-place. `NEW_NAME` is just the new name, not a full path.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
SOURCE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/ready/genAI_2/"
NEW_NAME    = "genAI_2_extracted"   # new name only (not a full path)
# ───────────────────────────────────────────────────────────────────────────

rename(SOURCE_PATH, NEW_NAME)

## 7 — Delete

⚠ **Permanent — no recycle bin.** You will be asked to type `yes` to confirm.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
TARGET_PATH = "/home/taiaburrahman/dataset_manager_pro/data/backup/genAI_2/"
# ───────────────────────────────────────────────────────────────────────────

delete(TARGET_PATH, confirm=True)   # confirm=False skips the prompt

## Appendix — Browse & Disk Usage

**`browse(path)`** — tree view with sizes and timestamps.  
**`disk_usage(path)`** — size ranking of every item in a directory.

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
BROWSE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/"
MAX_DEPTH   = 3      # how many folder levels to show
SHOW_HIDDEN = False  # include dot-files / dot-folders
# ───────────────────────────────────────────────────────────────────────────

browse(BROWSE_PATH, max_depth=MAX_DEPTH, show_hidden=SHOW_HIDDEN)

In [ ]:
# ── INPUT ──────────────────────────────────────────────────────────────────
USAGE_PATH = "/home/taiaburrahman/dataset_manager_pro/data/"
# ───────────────────────────────────────────────────────────────────────────

disk_usage(USAGE_PATH)